# 📖 Notebook 1: Data Modeling with Partition Keys

In relational databases, you model data around *entities and relationships*. In Cassandra, you model data around *queries*. The partition key is the single most important decision you make — it determines which node stores your data and how efficiently you can read it.

## Learning Objectives

By the end of this notebook, you'll understand:
- What a keyspace and table are in Cassandra
- How partition keys determine data placement
- Why query-driven modeling matters
- How to pick good vs. bad partition keys
- The Discord messages example: single vs. composite partition keys

## 🛠️ Setup

Start the Cassandra cluster first:

```bash
cd 03-technologies/databases/cassandra
docker compose up -d
```

Wait ~2 minutes for the cluster to be ready, then verify:

```bash
docker compose exec cassandra-node1 cqlsh -e "DESCRIBE CLUSTER"
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
from cassandra.cluster import Cluster
from cassandra.query import SimpleStatement
from tabulate import tabulate
import time

# Connect to the Cassandra cluster.
# We only specify node1 — the driver discovers the rest via gossip.
# max_schema_agreement_wait: after a CREATE TABLE, the schema has to reach
# every node before another node will answer a read against that table.
# The driver default (10s) is not enough on a freshly-formed 3-node ring,
# and the symptom is a confusing
#   ReadFailure ... INCOMPATIBLE_SCHEMA
# on the very first SELECT rather than on the DDL that caused it.
cluster = Cluster(["localhost"], port=9042, max_schema_agreement_wait=60)
session = cluster.connect()

# --- Make DDL deterministic on a multi-node ring -----------------------------
# Cassandra propagates schema changes by gossip. On this 3-node cluster a
# CREATE TABLE followed immediately by a read or write can reach a replica that
# has not seen the new schema yet, and the coordinator answers with a confusing
#   ReadFailure / WriteFailure ... INCOMPATIBLE_SCHEMA
# pointing at a random node. Waiting for every node to agree after each DDL
# statement removes that race, so these notebooks behave the same every run.
# LWT (Paxos) and QUORUM operations on a small local ring can exceed the
# driver default of 10s while the cluster is still warming up.
session.default_timeout = 30
_execute = session.execute


def _execute_and_settle(query, *args, **kwargs):
    """session.execute, hardened against schema gossip still settling.

    Two things happen here:

    1. After DDL we wait for every node to agree on the new schema.
    2. If a query still comes back INCOMPATIBLE_SCHEMA, we retry it. On a ring
       whose third node finished bootstrapping seconds ago, `wait_for_schema_
       agreement` can return while a replica is not yet serving the new table,
       and the coordinator answers Read/WriteFailure naming that node. It is
       transient, and retrying is exactly what a production client does with a
       transient coordinator failure.
    """
    import time as _time

    text = query if isinstance(query, str) else getattr(query, "query_string", "")
    is_ddl = text.strip().upper().startswith(("CREATE", "ALTER", "DROP", "TRUNCATE"))

    last = None
    for attempt in range(8):
        try:
            result = _execute(query, *args, **kwargs)
            if is_ddl:
                cluster.control_connection.wait_for_schema_agreement(wait_time=60)
            return result
        except Exception as exc:
            # INCOMPATIBLE_SCHEMA: a replica has not caught up with the new schema.
            # timed out / CAS operation timed out: a replica was too slow to answer.
            # Both are transient on a ring that has just booted; a real client
            # retries them too. Anything else is a genuine error -- re-raise it.
            transient = ("INCOMPATIBLE_SCHEMA" in str(exc)) or ("timed out" in str(exc).lower())
            if not transient:
                raise
            last = exc
            _time.sleep(2 * (attempt + 1))
    raise RuntimeError(
        f"Schema never settled after 8 retries. Last error: {last}\n"
        "Check `docker compose exec cassandra-node1 nodetool describecluster` -- "
        "all three nodes should report a single schema version."
    ) from last


session.execute = _execute_and_settle


def wait_for_cluster_ready(timeout=180.0):
    """Block until the ring can actually create a table and write to it at ALL.

    `docker compose up --wait` returns as soon as every node reports UN, but a
    node that has only just finished bootstrapping will still reject queries
    for a short while with INCOMPATIBLE_SCHEMA. Rather than sprinkle sleeps
    around, we prove readiness once: create a throwaway RF=3 table, write to it
    at consistency ALL (so every replica must answer), then drop it.
    """
    import time as _time

    from cassandra.query import SimpleStatement as _SimpleStatement
    from cassandra import ConsistencyLevel as _CL

    deadline = _time.time() + timeout
    last = None
    while _time.time() < deadline:
        try:
            _execute(
                "CREATE KEYSPACE IF NOT EXISTS readiness_probe WITH replication = "
                "{'class': 'SimpleStrategy', 'replication_factor': 3}"
            )
            cluster.control_connection.wait_for_schema_agreement(wait_time=60)
            _execute("CREATE TABLE IF NOT EXISTS readiness_probe.ping (id int PRIMARY KEY)")
            cluster.control_connection.wait_for_schema_agreement(wait_time=60)
            _execute(_SimpleStatement(
                "INSERT INTO readiness_probe.ping (id) VALUES (1)",
                consistency_level=_CL.ALL,
            ))
            _execute("DROP KEYSPACE readiness_probe")
            cluster.control_connection.wait_for_schema_agreement(wait_time=60)
            return
        except Exception as exc:  # replica still settling -- back off and retry
            last = exc
            _time.sleep(3)
    raise TimeoutError(
        f"Cassandra ring never became ready within {timeout}s. Last error: {last}\n"
        "Check `docker compose exec cassandra-node1 nodetool status` -- you need 3 UN nodes."
    )


wait_for_cluster_ready()
print("Ring ready: all 3 replicas accept schema changes and ALL-consistency writes")
# -----------------------------------------------------------------------------

print(f"Connected to cluster: {cluster.metadata.cluster_name}")
print(f"Nodes discovered: {list(cluster.metadata.all_hosts())}")

## Step 1: Creating a Keyspace

A **keyspace** is Cassandra's equivalent of a database in PostgreSQL. It's the top-level container
that holds tables and defines how data is replicated.

Think of it like this:
- PostgreSQL: `CREATE DATABASE my_app;`
- Cassandra: `CREATE KEYSPACE my_app WITH REPLICATION = {...};`

The key difference is that Cassandra requires you to specify a **replication strategy** upfront.
We'll use `SimpleStrategy` with a replication factor of 3 (one copy on each of our 3 nodes).

In [ ]:
# Create a keyspace with replication factor 3 (data copied to all 3 nodes)
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS demo
    WITH REPLICATION = {
        'class': 'SimpleStrategy',
        'replication_factor': 3
    }
""")

session.set_keyspace('demo')

# --- Read-your-writes on a 3-node ring ---------------------------------------
# The driver's default consistency level is LOCAL_ONE. With replication_factor
# 3 that means a write can be acknowledged by a single replica while the very
# next read is served by a different replica that has not received it yet -- so
# a read-after-write silently returns None and the cell dies on an attribute of
# NoneType. Nothing in this notebook is about that trade-off, so we take the
# safe side: QUORUM on both reads and writes gives R + W > N, which is exactly
# what makes read-your-writes hold. Notebook 3 sets a consistency level per
# statement to demonstrate the trade-off deliberately, and a per-statement
# level always overrides this default.
from cassandra import ConsistencyLevel

session.default_consistency_level = ConsistencyLevel.QUORUM

print("Keyspace 'demo' created and selected!")

> 💡 **Production note:** `SimpleStrategy` is fine for this single-datacenter lab, but in production use `NetworkTopologyStrategy` — it's rack- and datacenter-aware, so a rack failure doesn't take out multiple replicas. We'll see it in Notebook 3.

## Step 2: Your First Table — A Simple Partition Key

Let's create a simple `users` table. The **partition key** is `user_id`.

When you insert a row, Cassandra does this:
1. Takes the partition key value (e.g., `user_id = 42`)
2. Hashes it using a consistent hash function → produces a **token**
3. Finds which node owns that token range on the **token ring**
4. Stores the row on that node (and its replicas)

```
user_id = 42  →  hash(42) = token 7823...  →  Node 2 owns that range  →  stored on Node 2
```

This means **all data with the same partition key lives on the same node**. This is a very
important property that drives how we model data.

In [ ]:
# Create a simple users table — user_id is the partition key
session.execute("""
    CREATE TABLE IF NOT EXISTS users (
        user_id bigint,
        username text,
        email text,
        created_at timestamp,
        PRIMARY KEY (user_id)
    )
""")

print("Table 'users' created!")
print("Partition key: user_id")
print("Clustering keys: (none)")

In [ ]:
from datetime import datetime

# Insert some users
users = [
    (1, 'alice', 'alice@example.com', datetime(2024, 1, 15)),
    (2, 'bob', 'bob@example.com', datetime(2024, 2, 20)),
    (3, 'charlie', 'charlie@example.com', datetime(2024, 3, 10)),
    (4, 'diana', 'diana@example.com', datetime(2024, 4, 5)),
    (5, 'eve', 'eve@example.com', datetime(2024, 5, 1)),
]

insert_stmt = session.prepare("""
    INSERT INTO users (user_id, username, email, created_at)
    VALUES (?, ?, ?, ?)
""")

for user in users:
    session.execute(insert_stmt, user)

print(f"Inserted {len(users)} users")

## Step 3: Querying by Partition Key — Fast!

Cassandra is **extremely fast** when you query by partition key because it knows *exactly*
which node to ask. No scanning, no searching — just hash and go.

This is the golden rule:
> **Always query by partition key.** Queries without a partition key hit ALL nodes (full cluster scan).

In [ ]:
# GOOD query: specify the partition key → goes to exactly one node
rows = session.execute("SELECT * FROM users WHERE user_id = 1")
for row in rows:
    print(f"Found user: {row.username} ({row.email})")

print("\n✅ This query is fast — Cassandra hashes user_id=1 and goes directly to the right node.")

In [ ]:
# BAD query: no partition key → Cassandra must scan ALL nodes
# This requires ALLOW FILTERING, which is Cassandra warning you this is expensive
try:
    rows = session.execute("SELECT * FROM users WHERE username = 'alice'")
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nCassandra rejected this query because 'username' is not the partition key.")
    print("Without a partition key, Cassandra would have to scan every node — that's slow and expensive.")
    print("\nYou CAN force it with 'ALLOW FILTERING', but that's almost always a bad idea in production.")

## Step 4: See Where Data Lives — Token Distribution

Let's peek behind the curtain and see the actual token values Cassandra assigns.
Each partition key value maps to a **token** (a big integer), and each node owns a range of tokens.

In [ ]:
# Show the token (hash value) for each user's partition key.
# NOTE: we alias to `partition_token`, not `token` -- `token` is a reserved
# word in CQL, so `AS token` is a syntax error.
rows = session.execute("SELECT user_id, username, TOKEN(user_id) AS partition_token FROM users")

table_data = []
for row in rows:
    table_data.append([row.user_id, row.username, row.partition_token])

print(tabulate(table_data, headers=["user_id", "username", "token (hash)"], tablefmt="grid"))
print("\nEach token maps to a position on Cassandra's token ring.")
print("The node that 'owns' that position stores the data.")

## Step 5: Query-Driven Modeling — The Discord Example

In relational databases, you start with entities (users, messages, channels) and normalize them.
In Cassandra, you start with **queries** and design tables to serve them.

### The Problem
Discord needs to store billions of messages. The most common query is:
> "Give me the most recent messages in channel X"

### First Attempt
Let's model this with `channel_id` as the partition key:

In [ ]:
# Discord's first approach: partition by channel_id
session.execute("""
    CREATE TABLE IF NOT EXISTS messages_v1 (
        channel_id bigint,
        message_id bigint,
        author_id bigint,
        content text,
        PRIMARY KEY (channel_id, message_id)
    ) WITH CLUSTERING ORDER BY (message_id DESC)
""")

print("Table: messages_v1")
print("Partition key:  channel_id    → all messages for a channel live on ONE node")
print("Clustering key: message_id    → messages sorted by ID (newest first) within the partition")
print("\nThis is great for the query 'get recent messages in channel X'!")

In [ ]:
import random

# Simulate inserting messages into a busy channel
insert_msg = session.prepare("""
    INSERT INTO messages_v1 (channel_id, message_id, author_id, content)
    VALUES (?, ?, ?, ?)
""")

channel_id = 100  # a single channel
for i in range(1, 21):
    session.execute(insert_msg, (
        channel_id,
        i,                           # message_id (acts like a snowflake ID)
        random.choice([1, 2, 3]),    # random author
        f"Message number {i} in the channel"
    ))

print(f"Inserted 20 messages into channel {channel_id}")

# Query recent messages — this hits a SINGLE partition (fast!)
rows = session.execute("""
    SELECT message_id, author_id, content
    FROM messages_v1
    WHERE channel_id = 100
    LIMIT 5
""")

print("\nMost recent 5 messages:")
table_data = [[r.message_id, r.author_id, r.content] for r in rows]
print(tabulate(table_data, headers=["message_id", "author_id", "content"], tablefmt="grid"))

## Step 6: The Partition Size Problem

The `messages_v1` schema works well... until a channel gets **very** busy.

The problem: ALL messages for a channel live in ONE partition. Popular Discord channels
can have millions of messages, creating **huge partitions** that:
- Slow down reads (Cassandra scans more data)
- Slow down compaction (more data to merge)
- Grow forever (partitions never shrink)

### The Fix: Composite Partition Keys with Time Buckets

Discord's solution: add a **time bucket** to the partition key. Each bucket covers 10 days.
This splits a channel's messages across multiple partitions by time.

```
Before: partition = (channel_id)              → 1 huge partition per channel
After:  partition = (channel_id, bucket)      → many smaller partitions per channel
```

In [ ]:
# Discord's improved approach: composite partition key with time bucket
session.execute("""
    CREATE TABLE IF NOT EXISTS messages_v2 (
        channel_id bigint,
        bucket int,
        message_id bigint,
        author_id bigint,
        content text,
        PRIMARY KEY ((channel_id, bucket), message_id)
    ) WITH CLUSTERING ORDER BY (message_id DESC)
""")

print("Table: messages_v2")
print("Partition key:  (channel_id, bucket)  → messages split into time windows")
print("Clustering key: message_id             → sorted newest-first within each bucket")
print("\nNote the double parentheses: ((channel_id, bucket)) means BOTH columns")
print("together form the partition key. This is a COMPOSITE partition key.")

In [ ]:
# Helper: compute a 10-day bucket from a message_id (simulated as days since epoch)
def get_bucket(day_number):
    """Bucket = 10-day windows. Day 0-9 → bucket 0, day 10-19 → bucket 1, etc."""
    return day_number // 10

# Insert messages spanning multiple buckets
insert_v2 = session.prepare("""
    INSERT INTO messages_v2 (channel_id, bucket, message_id, author_id, content)
    VALUES (?, ?, ?, ?, ?)
""")

channel_id = 100
for day in range(0, 30):  # 30 days of messages → 3 buckets
    bucket = get_bucket(day)
    msg_id = day * 100 + random.randint(1, 99)  # unique-ish ID
    session.execute(insert_v2, (
        channel_id,
        bucket,
        msg_id,
        random.choice([1, 2, 3]),
        f"Message on day {day} (bucket {bucket})"
    ))

print(f"Inserted 30 messages across 3 buckets (0, 1, 2)")

# Query the most recent bucket (bucket 2 = days 20-29)
current_bucket = get_bucket(29)
rows = session.execute(f"""
    SELECT message_id, bucket, content
    FROM messages_v2
    WHERE channel_id = {channel_id} AND bucket = {current_bucket}
    LIMIT 5
""")

print(f"\nRecent messages from bucket {current_bucket}:")
table_data = [[r.message_id, r.bucket, r.content] for r in rows]
print(tabulate(table_data, headers=["message_id", "bucket", "content"], tablefmt="grid"))

## Step 7: Comparing Partition Key Choices

Let's verify that different partition key designs actually distribute data across different nodes.
We'll look at the tokens to see the distribution.

In [ ]:
# v1: All messages for channel 100 have the SAME token (same partition)
rows = session.execute("""
    SELECT channel_id, message_id, TOKEN(channel_id) AS partition_token
    FROM messages_v1
    WHERE channel_id = 100
    LIMIT 5
""")

print("=== messages_v1: single partition key (channel_id) ===")
table_data = [[r.channel_id, r.message_id, r.partition_token] for r in rows]
print(tabulate(table_data, headers=["channel_id", "message_id", "token"], tablefmt="grid"))
print("↑ All tokens are IDENTICAL — all data on the same node\n")

# v2: Messages for channel 100 across buckets have DIFFERENT tokens
rows = session.execute("""
    SELECT channel_id, bucket, message_id, TOKEN(channel_id, bucket) AS partition_token
    FROM messages_v2
    LIMIT 10
    ALLOW FILTERING
""")

print("=== messages_v2: composite partition key (channel_id, bucket) ===")
table_data = [[r.channel_id, r.bucket, r.message_id, r.partition_token] for r in rows]
print(tabulate(table_data, headers=["channel_id", "bucket", "message_id", "token"], tablefmt="grid"))
print("↑ Different buckets get DIFFERENT tokens — data spread across nodes!")

## 🧠 Key Takeaways

1. **Partition key = data placement**. Rows with the same partition key live on the same node.

2. **Always query by partition key**. Queries without it require a full cluster scan (ALLOW FILTERING).

3. **Model for queries, not entities**. Ask "what queries does my app need?" before designing tables.

4. **Watch partition size**. If a partition can grow forever, add a bucketing column to the partition key.

5. **Composite partition keys** let you split hot partitions across nodes.

## ➡️ Next Notebook

In Notebook 2, we'll explore **clustering columns** — how to sort data within a partition and
build "wide rows" that serve complex queries from a single read.

In [ ]:
# Clean up
cluster.shutdown()
print("Connection closed.")